# Step 2: Generate summary data...with functions 😎

Generate summary data as html tables
* create summary for Lichert scale responses (i.e., pivot table of responses)
* get comments
* split topics (comma-separated responses)
* combine summary data

## 1. Set variables

In [1]:
# files to open
access_eval = 'ACCESS Advising Evaluation (Responses).xlsx'
soc_eval = 'SOC Advising (Responses).xlsx'

# time variables
filter_year = 2025
filter_month = 11

# questions to remove
# November 2025 survey questions changed and Google Sheets was reindexed.
# Starting December 2025, questions do not need to be removed.

# text to remove from each question
eval_remove_text = ['Please evaluate your advising session',
                    'Please evaluate the advisor',
                    'Overall advising appointment',
                    '[',
                    ']',
                    '.',
                    ':',
                    'Check all that apply.']

# text to replace
comments_replace = ['Please feel free to make additional comments of the above questions', 'Additional comments?']

# Likert scale values
likert_scale_access1 = ['Strongly Agree (5)', 'Agree (4)', 'Neutral (3)', 'Disagree (2)', 'Strongly Disagree (1)', 'Not Applicable (0)', 'No Response']
likert_scale_access2 = ['Excellent (5)', 'Good (4)', 'Average (3)', 'Fair (2)', 'Poor (1)', 'No Response']
likert_scale_soc1 = ['Strongly agree', 'Agree', 'Neutral', 'Disagree', 'Strongly Disagree', 'No Response']
likert_scale_soc2 = ['Excellent', 'Good', 'Average', 'Fair', 'Poor', 'No Response']

# meeting types
meeting_types = ['Via email', 'Via phone', 'Via Zoom', 'In person']

## 2. Load and clean data (from Step 1)

In [2]:
# load libraries
import pandas as pd
from IPython.display import display, HTML  # library to display html in notebook

In [3]:
# load data
access_df = pd.read_excel(access_eval)
soc_df = pd.read_excel(soc_eval)

In [4]:
# clean data
def clean_data(df, text_remove_list, text_replace_list):
    for i in text_remove_list:
        df.columns = df.columns.str.replace(i, '')

    for i in text_replace_list:
        df.columns = df.columns.str.replace(i, 'Comments')

    df.columns = df.columns.str.strip()

    df = df[df['Timestamp'].dt.year == filter_year]
    df = df[df['Timestamp'].dt.month == filter_month]

    df = df.fillna('No Response')
    
    return df

# clean data
access_df = clean_data(access_df, eval_remove_text, comments_replace)
soc_df = clean_data(soc_df, eval_remove_text, comments_replace)

In [5]:
# check data
soc_df

,Timestamp,Advisor,How did you meet with your advisor?,What did your meeting include? Check all that apply,I was satisfied with how my advisor handled my questions,It was easy to talk to / connect with my advisor,My overall evaluation of this advising meeting is,My overall evaluation of this advisor is,Comments
850,2025-11-05 09:13:29.903,Barbara Joyce,Via Zoom,"Internships, Career information, Academic Plan...",Strongly agree,Strongly agree,Excellent,Excellent,No Response
851,2025-11-22 10:23:13.182,Dawn Nishida,Via Zoom,"Registration Questions, Academic Planning, BAM...",Strongly agree,Strongly agree,Excellent,Excellent,No Response
852,2025-11-24 14:50:37.496,Barbara Joyce,Via Zoom,Registration Questions,Strongly agree,Strongly agree,Excellent,Excellent,I feel a sense of comfort when I’m meeting wit...
853,2025-11-25 00:18:38.081,Barbara Joyce,Via Zoom,"Registration Questions, Graduation, Advice abo...",Strongly agree,Strongly agree,Excellent,Excellent,No Response
854,2025-11-27 10:58:45.681,Barbara Joyce,Via Zoom,"Registration Questions, Mandatory Advising, Ac...",Strongly agree,Strongly agree,Excellent,Excellent,No Response


## 3. Filter data for a single advisor

In [6]:
# filter data for a single advisor
#advisor = 'Marilou'
#filtered_advisor = access_df[access_df['Advisor'].str.startswith(advisor)]

def filtered_advisor_df (advisor_name, df_name):
    filtered_advisor = df_name[df_name['Advisor'].str.startswith(advisor_name)]

    return filtered_advisor

In [7]:
# check data
#filtered_advisor
#filtertest

## 4. Functions to generate summary data

In [8]:
# function to create summary data for Likert scale responses
# def function_name(var1, var2):
#    code to create summary data
#    return html_table

def create_summary_data(col_beg, col_end, scale, filtered_advisor):
    question_score = []

    # get list of questions
    questions = []
    questions = filtered_advisor.columns[col_beg:col_end]

    number_responses = len(filtered_advisor)

    for i in questions:
        for j in scale:
            count = (filtered_advisor[i] == j).sum()
            percent = count / number_responses * 100
            display_count = f'{count}<br>{percent:.1f}%'
            question_score.append({
                'Question': i,
                'Score': j,
                'Count': display_count
            })

    df = pd.DataFrame(question_score)
    df = df.pivot(index='Question', columns='Score', values='Count')

    df = df[scale].loc[questions]
    df.columns.name = None

    html_likert = df.to_html(escape=False)
    
    return html_likert

In [9]:
# function to get comments
# def function_name(var1, var2):
#    code to get comments
#    return html_table

def create_comments_data(col_beg, col_end, filtered_advisor):
    comments = filtered_advisor.iloc[:, col_end]

    comments = comments[comments != 'No Response']

    commentsdf = pd.DataFrame(comments)

    html_comments = commentsdf.to_html(index=False)

    if len(html_comments) < 1:
        html_comments = ''

    return html_comments

In [10]:
# function to split comma-separated responses
# def function_name(var1, var2):
#    code to split responses
#    return html_table

def soc_split_topics(meeting_col, rename_col, filtered_advisor):
    soc_topics = []

    meeting = filtered_advisor.columns[meeting_col]
    responses = filtered_advisor[meeting].values

    for i in responses:
        # replace response that contains a comma
        i = i.replace('Major information, options', 'Major information/options')
        i = i.replace(', ', ',')

        # split values by comma
        split = i.split(',')

        # add topic to list
        for j in split:
            soc_topics.append(j)

    df = pd.DataFrame(soc_topics)

    summary = df.value_counts().reset_index()
    summary = summary.rename(columns={rename_col: 'Topics'})
    summary = summary.rename(columns={1: 'Number of Responses'})

    html_topics = summary.to_html(index=False)

    html_topics = html_topics + '<p>One student can select more than one topic during a meeting.</p>'

    return html_topics

In [11]:
# meeting types function for advisor

def soc_meeting_mode(meeting_mode_col, rename_col, filtered_advisor):
    # 2 column
    meeting_mode = filtered_advisor.columns[meeting_mode_col]
    mode_responses = filtered_advisor[meeting_mode].values

    df = pd.DataFrame(mode_responses)
    meeting_types_count = df.value_counts().reset_index()
    meeting_types_count = meeting_types_count.rename(columns={rename_col: 'How did you meet with your advisor?'})
    meeting_types_count = meeting_types_count.rename(columns={1: 'Number of Responses'})

    html_meeting_mode = meeting_types_count.to_html(index=False)
    
    return html_meeting_mode

In [14]:
# function to combine summary data (put html tables together)
# def function_name(var1, var2):
#    code to combine html tables together
#    return html_final
# html1 = html_likert
# html2 = html_comments
# html3 = html_topics
# html4 = html_meeting_types


def combine_summary_data(df_name, html1, html2, html3, html4):
    if df_name == 'soc_df':
        html_summary_data = html4 + html3 + html1 + html2
    else:
        html_summary_data = html1 + html2

    return html_summary_data

In [15]:
# call function

# loop through both lists, shift the columns for each type of likert scales
# loop for each advisor

html_file = ''
html_topics = ''
html_meeting_mode = ''

# filtered_advisor = filtered_advisor_df ('Marilou', access_df)

html_header = '<!DOCTYPE html> <html lang="en"> <head>   <title>Monthly Advisor Evals</title>'
html_header = html_header + '<meta charset="utf-8">   <meta name="viewport" content="width=device-width, initial-scale=1">'
html_header = html_header + '<link href="https://cdn.jsdelivr.net/npm/bootstrap@5.3.3/dist/css/bootstrap.min.css" rel="stylesheet">'
html_header = html_header + '<script src="https://cdn.jsdelivr.net/npm/bootstrap@5.3.3/dist/js/bootstrap.bundle.min.js"></script>'
html_header = html_header + '</head><body>'

# accessdf
advisors_list = access_df['Advisor'].unique()

for i in advisors_list:
    filtered_advisor = filtered_advisor_df(i, access_df)

    html_header = html_header + '<h1>Monthly Evals for ' + i + '</h1>' + '<h5>Date: datevariable'+ '</h5>'

    # likerts
    html_likert = create_summary_data(2, 9, likert_scale_access1, filtered_advisor)
    html_likert = html_likert + create_summary_data(9, 11, likert_scale_access2, filtered_advisor)

    # comments
    html_comments = create_comments_data(0, -1, filtered_advisor)

    # combined summary_data
    html_summary_data = combine_summary_data('access_df', html_likert, html_comments, html_topics, html_meeting_mode)

    html_file = html_file + html_header + html_summary_data + '<div style="page-break-after: always;"></div>'
    html_header = ''

# socdf
advisors_list = soc_df['Advisor'].unique()

for i in advisors_list:
    filtered_advisor = filtered_advisor_df(i, soc_df)

    html_header = '<h1>Monthly Evals for ' + i + '</h1>' + '<h5>Date: datevariable'+ '</h5>'

    # likerts
    html_likert = create_summary_data(4, 6, likert_scale_soc1, filtered_advisor)
    html_likert = html_likert + create_summary_data(6, 8, likert_scale_soc2, filtered_advisor)

    # comments
    html_comments = create_comments_data(0, -1, filtered_advisor)

    # topics
    html_topics = soc_split_topics(3, 0, filtered_advisor)

    # meeting mode
    html_meeting_mode = soc_meeting_mode(2, 0, filtered_advisor)

    # combined summary_data
    html_summary_data = combine_summary_data('soc_df', html_likert, html_comments, html_topics, html_meeting_mode)

    html_file = html_file + html_header + html_summary_data + '<div style="page-break-after: always;"></div>'

html_file = html_file + '</body></html>'
#display(HTML(html_file))

display(HTML(html_file))

# check data
file_path = 'eval.html'
with open(file_path, 'w', encoding='utf-8') as file:
    file.write(html_file)

## 👏 You made really nice functions, yay!